# 220 · Smoke Test Sanity Check (2 samples, 2 insertions)

**Goal:** Verify the full pipeline works end-to-end before launching the production grid.

Smoke test config: `i=2, c=2, mb=5, vol=75, direction=buy, n_samples=2`

With only 1 grid point and 2 samples we **cannot** compute beta (need multiple Q).
Instead we check:
1. Data integrity — all 9 models loaded, correct shapes
2. Mid-price trajectories — impact visible at aggressive order positions
3. Spread dynamics — spread widens at insertion, recovers
4. Conditioning vs generation — price continuity at the boundary
5. Model comparison — overlay all 9 models on one plot
6. Message-level diagnostics — event types, order flow

In [ ]:
import numpy as np
import pandas as pd
import yaml, re, math, json
from pathlib import Path
from collections import OrderedDict
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
TICK_SIZE = 100
SMOKE_DIR = Path('../data/evalsequences/smoke_test')
STOCK = 'GOOG'

MODEL_KEYS = ['LobS5', 'S5-150M', 'S5-4K', 'S5-360M', 'CGAN',
              'ZeroInsertions', 'Historic', 'Heuristic', 'CST']

MODEL_META = OrderedDict([
    ('LobS5',          dict(color='#1B9E77', dash='solid',   marker='circle')),
    ('S5-150M',         dict(color='#D95F02', dash='solid',   marker='triangle-down')),
    ('S5-4K',           dict(color='#7570B3', dash='solid',   marker='star')),
    ('S5-360M',         dict(color='#E7298A', dash='solid',   marker='hexagon')),
    ('CGAN',            dict(color='#8B4513', dash='longdash',marker='square')),
    ('ZeroInsertions',  dict(color='#66A61E', dash='dot',     marker='diamond-open')),
    ('Historic',        dict(color='#999999', dash='dot',     marker='x')),
    ('Heuristic',       dict(color='#D4A017', dash='dashdot', marker='diamond')),
    ('CST',             dict(color='#E07B39', dash='dash',    marker='triangle-up')),
])

TEMPLATE = 'plotly_white'
FONT = dict(family='Times New Roman, serif', size=14)
W, H = 1080, 450

---
## 1. Load Data

In [ ]:
def find_latest_exp(model_name):
    """Find the latest exp_* folder for a model."""
    model_dir = SMOKE_DIR / model_name / STOCK
    if not model_dir.exists():
        return None
    exps = sorted(model_dir.glob('exp_*'), key=lambda p: p.stat().st_mtime, reverse=True)
    return exps[0] if exps else None


def load_experiment(exp_path):
    """Load all data from one experiment folder."""
    result = dict(path=exp_path, config={}, aggr_idx=[], samples=[])

    # Config
    cfg_file = exp_path / 'config.yaml'
    if cfg_file.exists():
        with open(cfg_file) as f:
            result['config'] = yaml.safe_load(f)

    # Aggressive indices
    aggr_file = exp_path / 'aggressive_indices.csv'
    if aggr_file.exists():
        result['aggr_idx'] = np.loadtxt(aggr_file, dtype=int).tolist()
        if isinstance(result['aggr_idx'], int):
            result['aggr_idx'] = [result['aggr_idx']]

    # Load samples (matched message + orderbook pairs)
    gen_dir = exp_path / 'data_gen'
    cond_dir = exp_path / 'data_cond'
    if not gen_dir.exists():
        return result

    ob_files = sorted(gen_dir.glob(f'{STOCK}_*_orderbook_*_gen_id_0.csv'))
    for ob_file in ob_files:
        # Extract sample id from filename
        m = re.search(r'real_id_(\d+)_gen_id', ob_file.name)
        if not m:
            continue
        sample_id = m.group(1)
        date_m = re.search(r'(\d{4}-\d{2}-\d{2})', ob_file.name)
        date = date_m.group(1) if date_m else '?'

        msg_file = ob_file.parent / ob_file.name.replace('_orderbook_', '_message_')
        sample = dict(id=sample_id, date=date)

        try:
            sample['gen_book'] = pd.read_csv(ob_file, header=None).values
            sample['gen_msg']  = pd.read_csv(msg_file, header=None).values if msg_file.exists() else None
        except Exception as e:
            print(f'  WARN: {ob_file.name}: {e}')
            continue

        # Conditioning data
        cond_ob = cond_dir / f'{STOCK}_{date}_orderbook_real_id_{sample_id}.csv'
        cond_msg = cond_dir / f'{STOCK}_{date}_message_real_id_{sample_id}.csv'
        sample['cond_book'] = pd.read_csv(cond_ob, header=None).values if cond_ob.exists() else None
        sample['cond_msg']  = pd.read_csv(cond_msg, header=None).values if cond_msg.exists() else None

        result['samples'].append(sample)

    return result


# Load all 9 models
DATA = OrderedDict()
print(f"{'Model':<20} {'Exp folder':<35} {'Samples':>8} {'Gen msgs':>10} {'Aggr idx'}")
print('-' * 95)
for model in MODEL_KEYS:
    exp = find_latest_exp(model)
    if exp is None:
        print(f'{model:<20} NOT FOUND')
        continue
    d = load_experiment(exp)
    DATA[model] = d
    n_msgs = d['samples'][0]['gen_book'].shape[0] if d['samples'] else 0
    print(f"{model:<20} {exp.name:<35} {len(d['samples']):>8} {n_msgs:>10} {d['aggr_idx']}")

print(f'\nLoaded {len(DATA)} / {len(MODEL_KEYS)} models')

---
## 2. Data Integrity Checks

In [ ]:
def midprice(book_arr):
    """Mid-price from L2 book: (best_ask + best_bid) / 2."""
    ask = book_arr[:, 0].astype(float)
    bid = book_arr[:, 2].astype(float)
    mid = (ask + bid) / 2.0
    mid[mid <= 0] = np.nan
    return mid


def spread(book_arr):
    """Bid-ask spread in ticks."""
    ask = book_arr[:, 0].astype(float)
    bid = book_arr[:, 2].astype(float)
    s = (ask - bid) / TICK_SIZE
    s[(ask <= 0) | (bid <= 0)] = np.nan
    return s


print(f"{'Model':<18} {'Samp':>4}  {'Gen rows':>8} {'Cond rows':>9}  "
      f"{'Mid start':>12} {'Mid end':>12} {'Spread mean':>12} {'Event types'}")
print('-' * 115)
for model, d in DATA.items():
    for s in d['samples']:
        book = s['gen_book']
        msg  = s['gen_msg']
        mid = midprice(book)
        sp = spread(book)
        cond_rows = s['cond_book'].shape[0] if s['cond_book'] is not None else 0

        # Event type distribution
        if msg is not None:
            etypes, counts = np.unique(msg[:, 1].astype(int), return_counts=True)
            et_str = ' '.join(f'{e}:{c}' for e, c in zip(etypes, counts))
        else:
            et_str = '?'

        print(f"{model:<18} {s['id']:>4}  {book.shape[0]:>8} {cond_rows:>9}  "
              f"{np.nanmean(mid[:1]):>12,.0f} {np.nanmean(mid[-1:]):>12,.0f} "
              f"{np.nanmean(sp):>12.1f} {et_str}")

---
## 3. Mid-Price Trajectories (per model, all samples)

In [ ]:
n_models = len(DATA)
n_cols = min(4, n_models)
n_rows = math.ceil(n_models / n_cols)

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=list(DATA.keys()),
                    horizontal_spacing=0.05, vertical_spacing=0.12)

for idx, (model, d) in enumerate(DATA.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    meta = MODEL_META.get(model, {})
    aggr_idx = d['aggr_idx']

    for si, s in enumerate(d['samples']):
        mid = midprice(s['gen_book'])
        # Normalize to ticks from start
        mid_ticks = (mid - np.nanmean(mid[:1])) / TICK_SIZE
        x = np.arange(len(mid_ticks))

        fig.add_trace(go.Scatter(
            x=x, y=mid_ticks, mode='lines+markers',
            marker=dict(size=4, color=meta.get('color', '#888')),
            line=dict(color=meta.get('color', '#888'), width=1.5),
            name=f"sample {s['id']}", showlegend=(idx == 0),
            opacity=0.7 + 0.3 * si,
        ), row=row, col=col)

    # Mark aggressive order positions
    for ai in aggr_idx:
        fig.add_vline(x=ai, line_dash='dash', line_color='red',
                      line_width=1, opacity=0.6, row=row, col=col)

fig.update_layout(
    title='Mid-Price Trajectories (ticks from start, red = aggressive order)',
    template=TEMPLATE, font=FONT, width=W, height=H * n_rows * 0.7,
    showlegend=False)
fig.update_yaxes(title_text='ticks', col=1)
fig.update_xaxes(title_text='msg #', row=n_rows)
fig.show()

---
## 4. Spread Dynamics

In [ ]:
fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=list(DATA.keys()),
                    horizontal_spacing=0.05, vertical_spacing=0.12)

for idx, (model, d) in enumerate(DATA.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    meta = MODEL_META.get(model, {})
    aggr_idx = d['aggr_idx']

    for si, s in enumerate(d['samples']):
        sp = spread(s['gen_book'])
        fig.add_trace(go.Scatter(
            x=np.arange(len(sp)), y=sp, mode='lines+markers',
            marker=dict(size=4, color=meta.get('color', '#888')),
            line=dict(color=meta.get('color', '#888'), width=1.5),
            showlegend=False, opacity=0.7 + 0.3 * si,
        ), row=row, col=col)

    for ai in aggr_idx:
        fig.add_vline(x=ai, line_dash='dash', line_color='red',
                      line_width=1, opacity=0.6, row=row, col=col)

fig.update_layout(
    title='Bid-Ask Spread (ticks, red = aggressive order)',
    template=TEMPLATE, font=FONT, width=W, height=H * n_rows * 0.7,
    showlegend=False)
fig.update_yaxes(title_text='spread', col=1)
fig.update_xaxes(title_text='msg #', row=n_rows)
fig.show()

---
## 5. Conditioning → Generation Continuity

In [ ]:
# Show last 20 conditioning msgs + all generated msgs for first sample of each model
COND_TAIL = 20

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=list(DATA.keys()),
                    horizontal_spacing=0.05, vertical_spacing=0.12)

for idx, (model, d) in enumerate(DATA.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    meta = MODEL_META.get(model, {})
    if not d['samples']:
        continue
    s = d['samples'][0]

    # Conditioning tail
    if s['cond_book'] is not None:
        cond_mid = midprice(s['cond_book'])
        tail = cond_mid[-COND_TAIL:]
        x_cond = np.arange(-len(tail), 0)
        fig.add_trace(go.Scatter(
            x=x_cond, y=tail / TICK_SIZE, mode='lines',
            line=dict(color='gray', width=2), name='cond',
            showlegend=(idx == 0),
        ), row=row, col=col)

    # Generated
    gen_mid = midprice(s['gen_book'])
    x_gen = np.arange(len(gen_mid))
    fig.add_trace(go.Scatter(
        x=x_gen, y=gen_mid / TICK_SIZE, mode='lines+markers',
        marker=dict(size=4, color=meta.get('color', '#888')),
        line=dict(color=meta.get('color', '#888'), width=2),
        name='gen', showlegend=(idx == 0),
    ), row=row, col=col)

    # Boundary
    fig.add_vline(x=-0.5, line_dash='solid', line_color='black',
                  line_width=1, row=row, col=col)
    for ai in d['aggr_idx']:
        fig.add_vline(x=ai, line_dash='dash', line_color='red',
                      line_width=1, opacity=0.5, row=row, col=col)

fig.update_layout(
    title=f'Conditioning (gray) → Generation (color), sample 0, last {COND_TAIL} cond msgs',
    template=TEMPLATE, font=FONT, width=W, height=H * n_rows * 0.7)
fig.update_yaxes(title_text='mid / tick', col=1)
fig.update_xaxes(title_text='msg # (0 = gen start)', row=n_rows)
fig.show()

---
## 6. All Models Overlaid (normalized impact)

In [ ]:
fig = go.Figure()

for model, d in DATA.items():
    meta = MODEL_META.get(model, {})
    aggr_idx = d['aggr_idx']

    curves = []
    for s in d['samples']:
        mid = midprice(s['gen_book'])
        impact = (mid - np.nanmean(mid[:1])) / TICK_SIZE
        curves.append(impact)

    if not curves:
        continue
    # Pad to same length and average
    max_len = max(len(c) for c in curves)
    padded = np.full((len(curves), max_len), np.nan)
    for i, c in enumerate(curves):
        padded[i, :len(c)] = c
    avg = np.nanmean(padded, axis=0)

    fig.add_trace(go.Scatter(
        x=np.arange(max_len), y=avg, mode='lines+markers',
        name=model,
        line=dict(color=meta.get('color', '#888'),
                  dash=meta.get('dash', 'solid'), width=2.5),
        marker=dict(symbol=meta.get('marker', 'circle'), size=5),
    ))

# Mark aggressive positions (same for all models with insertions)
for ai in [5, 11]:
    fig.add_vline(x=ai, line_dash='dash', line_color='red',
                  line_width=1, opacity=0.5,
                  annotation_text=f'aggr #{ai}' if ai == 5 else '')

fig.update_layout(
    title='Average Impact: All 9 Models (buy direction, 2 samples)',
    xaxis_title='Message #',
    yaxis_title='Mid-price impact (ticks)',
    template=TEMPLATE, font=FONT, width=W, height=H,
    legend=dict(x=0.01, y=0.99))
fig.show()

---
## 7. Message-Level Diagnostics

In [ ]:
EVENT_NAMES = {1: 'LO', 2: 'Cancel-Partial', 3: 'Cancel-Full', 4: 'MO/Exec', 5: 'Hidden-Exec'}

rows = []
for model, d in DATA.items():
    for s in d['samples']:
        msg = s['gen_msg']
        if msg is None:
            continue
        for et in sorted(EVENT_NAMES.keys()):
            count = int((msg[:, 1].astype(int) == et).sum())
            if count > 0:
                rows.append(dict(model=model, sample=s['id'],
                                 event_type=et, name=EVENT_NAMES[et], count=count))

et_df = pd.DataFrame(rows)
if not et_df.empty:
    pivot = et_df.groupby(['model', 'name'])['count'].sum().unstack(fill_value=0)
    print('Event type distribution (summed over samples):')
    print(pivot.to_string())
    print()

    fig = px.bar(et_df.groupby(['model', 'name'])['count'].sum().reset_index(),
                 x='model', y='count', color='name', barmode='stack',
                 title='Generated Event Types per Model',
                 color_discrete_sequence=px.colors.qualitative.Set2)
    fig.update_layout(template=TEMPLATE, font=FONT, width=W, height=H)
    fig.show()

---
## 8. L2 Book Depth at Aggressive Order

In [ ]:
def plot_l2_snapshot(book_row, title='', tick_size=TICK_SIZE):
    """Plot L2 book from one orderbook row (40 cols = 10 levels)."""
    n_levels = len(book_row) // 4
    asks_p = [book_row[4*k]   / tick_size for k in range(n_levels)]
    asks_v = [book_row[4*k+1] for k in range(n_levels)]
    bids_p = [book_row[4*k+2] / tick_size for k in range(n_levels)]
    bids_v = [book_row[4*k+3] for k in range(n_levels)]
    fig = go.Figure()
    fig.add_trace(go.Bar(x=asks_p, y=asks_v, name='Ask', marker_color='#EF553B', width=0.4))
    fig.add_trace(go.Bar(x=bids_p, y=[-v for v in bids_v], name='Bid', marker_color='#636EFA', width=0.4))
    fig.update_layout(title=title, template=TEMPLATE, font=FONT,
                      xaxis_title='Price (ticks)', yaxis_title='Volume',
                      width=520, height=350, barmode='overlay')
    return fig


# Show book snapshots before/after first aggressive order for LobS5
for model in ['LobS5', 'CGAN']:
    if model not in DATA or not DATA[model]['samples']:
        continue
    d = DATA[model]
    s = d['samples'][0]
    aggr = d['aggr_idx']
    if not aggr:
        continue
    ai = aggr[0]  # first aggressive order
    book = s['gen_book']

    if ai > 0 and ai < len(book):
        fig_before = plot_l2_snapshot(book[ai - 1], f'{model}: Book BEFORE aggr #{ai} (msg {ai-1})')
        fig_after  = plot_l2_snapshot(book[ai],     f'{model}: Book AFTER aggr #{ai} (msg {ai})')
        fig_before.show()
        fig_after.show()

---
## 9. Inter-Arrival Times

In [ ]:
fig = go.Figure()
for model, d in DATA.items():
    meta = MODEL_META.get(model, {})
    all_dt = []
    for s in d['samples']:
        msg = s['gen_msg']
        if msg is None or len(msg) < 2:
            continue
        times = msg[:, 0].astype(float)  # seconds with nanosecond fraction
        dt = np.diff(times)
        dt = dt[dt > 0]
        all_dt.extend(dt.tolist())

    if all_dt:
        all_dt = np.array(all_dt)
        fig.add_trace(go.Box(
            y=all_dt * 1000,  # ms
            name=model,
            marker_color=meta.get('color', '#888'),
            boxpoints='all', jitter=0.3, pointpos=-1.5,
        ))

fig.update_layout(
    title='Inter-Arrival Times (generated messages)',
    yaxis_title='dt (ms)', yaxis_type='log',
    template=TEMPLATE, font=FONT, width=W, height=H)
fig.show()

---
## 10. Summary Table

In [ ]:
rows = []
for model, d in DATA.items():
    cfg = d['config']
    n_samp = len(d['samples'])
    impacts, spreads_mean = [], []
    for s in d['samples']:
        mid = midprice(s['gen_book'])
        impact_ticks = (np.nanmean(mid[-1:]) - np.nanmean(mid[:1])) / TICK_SIZE
        impacts.append(impact_ticks)
        spreads_mean.append(np.nanmean(spread(s['gen_book'])))

    # Peak impact (at last aggressive order + 1)
    peak_impacts = []
    aggr = d['aggr_idx']
    if aggr:
        peak_idx = aggr[-1]  # after last aggressive
        for s in d['samples']:
            mid = midprice(s['gen_book'])
            if peak_idx < len(mid):
                peak_impacts.append((mid[peak_idx] - np.nanmean(mid[:1])) / TICK_SIZE)

    rows.append(dict(
        Model=model,
        n_samples=n_samp,
        n_msgs=d['samples'][0]['gen_book'].shape[0] if d['samples'] else 0,
        n_insertions=cfg.get('num_insertions', '?'),
        n_coolings=cfg.get('num_coolings', '?'),
        vol=cfg.get('order_volume', '?'),
        final_impact=f"{np.mean(impacts):.2f}" if impacts else '?',
        peak_impact=f"{np.mean(peak_impacts):.2f}" if peak_impacts else '?',
        mean_spread=f"{np.mean(spreads_mean):.1f}" if spreads_mean else '?',
    ))

summary = pd.DataFrame(rows)
print('\n=== Smoke Test Summary ===')
print(summary.to_string(index=False))
print()
print('Interpretation:')
print('  - final_impact: mid-price change from start to end (ticks), buy direction → expect positive for reactive models')
print('  - peak_impact: mid-price change at last aggressive order position')
print('  - ZeroInsertions: no aggressive orders → impact should be ~0 (random drift only)')
print('  - Historic: replay of real data → impact unrelated to injected orders')
print(f'\n  ⚠️  Only {n_samp} samples — numbers are noisy, this is a pipeline sanity check only.')